# Fine-Grained RAG Evaluation — Research Pipeline

Hybrid BM25 + multilingual E5 + RRF → Qwen → atomic claims → NLI → faithfulness/hallucination/completeness.

This notebook is a research prototype. Final paper results require human annotation, baselines and ablations.

In [ ]:
!pip -q install -r requirements.txt


In [ ]:
from src.config import load_config
from src.data import read_jsonl, build_corpus
from src.retriever import HybridRetriever
from src.generator import QwenGenerator
from src.claims import ClaimExtractor
from src.nli import ClaimEvidenceNLI
from src.evaluator import FineGrainedEvaluator

cfg = load_config()
cfg


In [ ]:
corpus = build_corpus(read_jsonl('data/sample_corpus.jsonl'))
questions = read_jsonl('data/sample_questions.jsonl')
retriever = HybridRetriever(corpus, cfg['retrieval']['dense_model'], cfg['retrieval']['rrf_k'])


In [ ]:
generator = QwenGenerator(
    cfg['generation']['model'],
    max_new_tokens=cfg['generation']['max_new_tokens'],
    temperature=cfg['generation']['temperature'],
    do_sample=cfg['generation']['do_sample']
)
claim_extractor = ClaimExtractor(generator, cfg['evaluation']['max_claims'])
nli = ClaimEvidenceNLI(cfg['nli']['model'])
evaluator = FineGrainedEvaluator(claim_extractor, nli, cfg['evaluation']['completeness_threshold'])


In [ ]:
for row in questions:
    evidence = retriever.hybrid_search(row['question'], cfg['retrieval']['top_k'])
    answer = generator.generate(row['question'], evidence)
    result = evaluator.evaluate(answer, evidence, row.get('reference_claims', []))
    print('\nQUESTION:', row['question'])
    print('ANSWER:', answer)
    print('EVIDENCE:', [x['id'] for x in evidence])
    print('CLAIMS:', result['claims'])
    print('FAITHFULNESS:', result['faithfulness'])
    print('HALLUCINATION:', result['hallucination_rate'])
    print('COMPLETENESS:', result['completeness'])


## Research experiments

1. Retrieval: BM25 vs dense E5 vs hybrid RRF.
2. Evaluator: compare automated scores with human labels.
3. Baselines: FActScore / ARES / RAGChecker where their implementations and compatible inputs are available.
4. Cross-language: English → Hindi → Malayalam → Tamil.
5. Ablations: remove claim decomposition, replace NLI, change retrieval, and test language-aware NLI.
6. Error analysis: claim extraction, retrieval, NLI, hallucination and completeness errors.